# Case Study: Tabular Credit Classification with MLPs

Companion notebook for the applied workflow in the MLP lecture notes.

Objectives:

- load the local credit train/test files from `data/`;
- build a reproducible preprocessing pipeline for mixed tabular data;
- compare logistic regression and scikit-learn MLP baselines;
- inspect loss curves and evaluation metrics;
- provide an optional PyTorch extension point when `torch` is installed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Load the local credit data

This notebook uses the local files `data/credtrain.txt` and `data/credtest.txt`. It does not depend on Colab paths.


In [ ]:
from pathlib import Path
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colnames = ['ESCT', 'NDEP', 'RENDA', 'TIPOR', 'VBEM', 'NPARC',
            'VPARC', 'TEL', 'IDADE', 'RESMS', 'ENTRADA', 'CLASSE']

train_path = Path('data/credtrain.txt')
test_path = Path('data/credtest.txt')

if not train_path.exists():
    train_path = Path('../../data/credtrain.txt')
    test_path = Path('../../data/credtest.txt')

train_df = pd.read_csv(train_path, sep='	', header=None, names=colnames)
test_df = pd.read_csv(test_path, sep='	', header=None, names=colnames)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
train_df.head()


## 2. Basic inspection


In [ ]:
summary = train_df.describe(include='all')
print(summary)

class_summary = pd.DataFrame({
    'train_count': train_df['CLASSE'].value_counts().sort_index(),
    'test_count': test_df['CLASSE'].value_counts().sort_index(),
})
class_summary


## 3. Preprocessing

We treat `ESCT`, `TIPOR`, and `TEL` as categorical indicators and scale the remaining numeric features. The pipeline is fit on training data only.


In [ ]:
target = 'CLASSE'
categorical_features = ['ESCT', 'TIPOR', 'TEL']
numeric_features = [c for c in colnames if c not in categorical_features + [target]]

X_train = train_df.drop(columns=[target])
y_train = train_df[target].astype(int)
X_test = test_df.drop(columns=[target])
y_test = test_df[target].astype(int)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numeric_features),
])

print('Categorical:', categorical_features)
print('Numeric:', numeric_features)


## 4. Baseline: logistic regression

Before fitting a neural network, establish a simple baseline.


In [ ]:
logreg_model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=2000)),
])
logreg_model.fit(X_train, y_train)
logreg_pred = logreg_model.predict(X_test)
logreg_prob = logreg_model.predict_proba(X_test)[:, 1]

print('Logistic regression accuracy:', accuracy_score(y_test, logreg_pred))
print('Logistic regression ROC AUC:', roc_auc_score(y_test, logreg_prob))
print(classification_report(y_test, logreg_pred))


## 5. Scikit-learn MLP

The MLP uses the same preprocessing object inside a pipeline. Early stopping reserves part of the training data internally for validation.


In [ ]:
mlp_model = Pipeline([
    ('preprocess', preprocess),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        solver='adam',
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42,
    )),
])

with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    mlp_model.fit(X_train, y_train)

mlp_pred = mlp_model.predict(X_test)
mlp_prob = mlp_model.predict_proba(X_test)[:, 1]

print('MLP accuracy:', accuracy_score(y_test, mlp_pred))
print('MLP ROC AUC:', roc_auc_score(y_test, mlp_prob))
print(classification_report(y_test, mlp_pred))


## 6. Compare models


In [ ]:
comparison = pd.DataFrame([
    {'model': 'LogisticRegression', 'accuracy': accuracy_score(y_test, logreg_pred), 'roc_auc': roc_auc_score(y_test, logreg_prob)},
    {'model': 'MLPClassifier', 'accuracy': accuracy_score(y_test, mlp_pred), 'roc_auc': roc_auc_score(y_test, mlp_prob)},
])
comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, logreg_pred, ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic regression')
ConfusionMatrixDisplay.from_predictions(y_test, mlp_pred, ax=axes[1], cmap='Blues')
axes[1].set_title('MLP')
plt.tight_layout()
plt.show()


## 7. MLP training curves


In [ ]:
mlp = mlp_model.named_steps['clf']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mlp.loss_curve_)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss')
axes[0].set_title('MLP loss curve')

if hasattr(mlp, 'validation_scores_'):
    axes[1].plot(mlp.validation_scores_)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation accuracy')
    axes[1].set_title('Early-stopping validation score')
else:
    axes[1].text(0.5, 0.5, 'No validation scores', ha='center')
plt.tight_layout()
plt.show()

print('Epochs run:', mlp.n_iter_)
print('Best validation score:', getattr(mlp, 'best_validation_score_', None))


## 8. Prediction threshold inspection

The default threshold is 0.5. Depending on business costs, a different threshold may be preferable.


In [ ]:
thresholds = np.linspace(0.1, 0.9, 9)
threshold_rows = []
for threshold in thresholds:
    pred = (mlp_prob >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'positive_rate': pred.mean(),
        'accuracy': accuracy_score(y_test, pred),
    })

pd.DataFrame(threshold_rows)


## 9. Optional PyTorch extension

If PyTorch is installed, this case study can be extended with a custom `nn.Module` using the transformed arrays from the same preprocessing pipeline. The cell below prepares those arrays and skips the PyTorch part when the dependency is missing.


In [ ]:
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print('PyTorch available:', TORCH_AVAILABLE)

X_train_processed = mlp_model.named_steps['preprocess'].transform(X_train)
X_test_processed = mlp_model.named_steps['preprocess'].transform(X_test)

print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)

if not TORCH_AVAILABLE:
    print('PyTorch is not installed. Use notebook 07 as the template for a PyTorch version of this case study.')


## Exercises

1. Change `hidden_layer_sizes` to `(64, 32)` and compare ROC AUC.
2. Increase `alpha` to `1e-2`. Does stronger L2 regularization help?
3. Choose a threshold based on a hypothetical cost where false positives are twice as expensive as false negatives.


## Takeaways

- Keep preprocessing inside the pipeline to avoid leakage.
- Compare MLPs against simple baselines.
- On tabular data, MLPs are not automatically better than linear models.
- Early stopping and regularization are important because tabular datasets are often small.
